# 第6周-Day5 Agent开发实战框架设计

🤖 **今日学习目标：掌握Agent框架设计的核心要素，能设计一个生产级Agent架构**

## 🎯 学习目标
- 理解Agent框架的**核心三要素**：LLM大脑 + 工具库 + 执行引擎
- 掌握Agent的**状态管理**策略：内存、数据库、混合模式
- 理解Agent的**错误处理与重试**机制
- 学会设计**可观测、可调试**的Agent系统

## 🔄 昨日复习
- Agent安全三大维度：**输入安全**（过滤恶意prompt）、**执行安全**（限制工具权限）、**输出安全**（审核生成内容）
- 可控性核心：**护栏（Guardrails）** 限制Agent行为边界
- 实战原则：最小权限原则 + 人工兜底 + 审计日志

In [ ]:
# 环境配置
import json
import time
from datetime import datetime
from typing import Dict, List, Any, Optional

import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False

print("使用字体:", font_name)

## 🏗️ 1. Agent框架核心三要素

一个完整的Agent框架由三个核心要素组成，缺一不可：

```
┌─────────────────────────────────┐
│         Agent 框架              │
│                                 │
│  🧠 LLM 大脑    决策与推理      │
│       ↓                         │
│  🔧 工具库      能力扩展        │
│       ↓                         │
│  ⚙️ 执行引擎    流程控制        │
│       ↓                         │
│  📊 可观测层    调试与监控      │
└─────────────────────────────────┘
```

In [ ]:
# Agent框架核心要素可视化
fig, axes = plt.subplots(1, 2, figsize=(16, 7))

# 左图：四大核心模块
ax1 = axes[0]
ax1.axis('off')
ax1.set_title('Agent 框架核心模块', fontsize=14, fontweight='bold')

modules = [
    ('🧠 LLM 大脑', '决策推理中心\n理解意图 → 选择工具 → 生成回复', '#E3F2FD', '#1565C0'),
    ('🔧 工具库', '能力扩展接口\nAPI调用/数据库/代码执行/搜索', '#FFF3E0', '#E65100'),
    ('⚙️ 执行引擎', '流程控制中心\nAgent Loop/状态机/编排', '#E8F5E9', '#2E7D32'),
    ('📊 可观测层', '调试监控接口\n日志/追踪/指标/告警', '#FCE4EC', '#C2185B'),
]

for i, (name, desc, bg, edge) in enumerate(modules):
    y = 0.85 - i * 0.22
    rect = patches.FancyBboxPatch((0.05, y-0.08), 0.9, 0.18,
                                    boxstyle="round,pad=0.05",
                                    facecolor=bg, edgecolor=edge, linewidth=2)
    ax1.add_patch(rect)
    ax1.text(0.1, y+0.04, name, fontsize=13, fontweight='bold', va='center')
    ax1.text(0.35, y+0.04, desc, fontsize=10, va='center', color='#333')
    if i < len(modules) - 1:
        ax1.annotate('', xy=(0.5, y-0.08), xytext=(0.5, y-0.04),
                    arrowprops=dict(arrowstyle='<-', color=edge, lw=1.5))

# 右图：数据流
ax2 = axes[1]
ax2.axis('off')
ax2.set_title('Agent 数据流向', fontsize=14, fontweight='bold')

flow = [
    (0.5, 0.95, '👤 用户输入', '#E3F2FD', '#1565C0'),
    (0.5, 0.82, '📝 系统提示词 + 对话历史', '#FFF9C4', '#F57F17'),
    (0.5, 0.69, '🧠 LLM 推理 → 是否调用工具？', '#E8F5E9', '#2E7D32'),
    (0.25, 0.53, '是 → 🔧 工具执行', '#FFF3E0', '#E65100'),
    (0.75, 0.53, '否 → 💬 直接回复', '#F3E5F5', '#6A1B9A'),
    (0.5, 0.38, '📋 结果整合', '#E0F7FA', '#00838F'),
    (0.5, 0.25, '🛡️ 安全检查（护栏）', '#FCE4EC', '#C2185B'),
    (0.5, 0.12, '📤 最终输出 + 日志记录', '#ECEFF1', '#37474F'),
]

for x, y, text, bg, edge in flow:
    rect = patches.FancyBboxPatch((x-0.25, y-0.04), 0.5, 0.08,
                                    boxstyle="round,pad=0.02",
                                    facecolor=bg, edgecolor=edge, linewidth=1.5)
    ax2.add_patch(rect)
    ax2.text(x, y, text, ha='center', va='center', fontsize=10)

plt.suptitle('Agent 框架设计全景', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 💾 2. 状态管理策略

Agent 需要记忆才能进行多轮对话。不同的状态管理策略适用于不同场景：

| 策略 | 存储 | 容量 | 适用场景 |
|------|------|------|----------|
| **内存** | Python dict/list | 小 | 单次会话、开发调试 |
| **文件** | JSON/SQLite | 中 | 持久化、小型应用 |
| **数据库** | PostgreSQL/Redis | 大 | 生产环境、多用户 |
| **向量库** | ChromaDB/Pinecone | 超大 | 长期记忆、RAG |

核心原则：**短期记忆用上下文窗口，长期记忆用外部存储**

In [ ]:
# 状态管理对比可视化
fig, ax = plt.subplots(1, 1, figsize=(14, 7))

strategies = ['内存\n(dict/list)', '文件\n(JSON/SQLite)', '数据库\n(PostgreSQL)', '向量库\n(ChromaDB)']
dimensions = {
    '容量': [2, 5, 9, 10],
    '速度': [10, 6, 5, 4],
    '持久性': [1, 7, 9, 8],
    '复杂度': [1, 4, 8, 7],
}

x = np.arange(len(strategies))
width = 0.18
colors = ['#42A5F5', '#66BB6A', '#FFA726', '#EF5350']

for i, (dim, values) in enumerate(dimensions.items()):
    bars = ax.bar(x + i * width, values, width, label=dim, color=colors[i], alpha=0.85)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.2, 
                str(int(bar.get_height())), ha='center', va='bottom', fontsize=9)

ax.set_xlabel('状态管理策略')
ax.set_ylabel('评分 (1-10)')
ax.set_title('Agent 状态管理策略对比', fontsize=14, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(strategies, fontsize=11)
ax.legend(fontsize=11)
ax.set_ylim(0, 12)
ax.grid(axis='y', alpha=0.3)

# 添加推荐标注
ax.annotate('开发调试首选', xy=(0, 10), fontsize=10, color='#1565C0', fontweight='bold',
            ha='center', bbox=dict(boxstyle='round', facecolor='#E3F2FD'))
ax.annotate('生产环境首选', xy=(2, 10), fontsize=10, color='#2E7D32', fontweight='bold',
            ha='center', bbox=dict(boxstyle='round', facecolor='#E8F5E9'))
ax.annotate('长期记忆首选', xy=(3, 10), fontsize=10, color='#6A1B9A', fontweight='bold',
            ha='center', bbox=dict(boxstyle='round', facecolor='#F3E5F5'))

plt.tight_layout()
plt.show()

## 🔁 3. 错误处理与重试机制

真实环境中工具调用一定会失败。健壮的Agent框架必须处理：

- **网络超时**：API调用超时怎么办？
- **参数错误**：LLM生成的参数不合法怎么办？
- **服务不可用**：工具所在服务挂了怎么办？
- **结果异常**：工具返回了意外格式的数据怎么办？

核心策略：**指数退避重试 + 优雅降级 + 人工兜底**

In [ ]:
# 错误处理与重试机制演示
import time

class RobustToolExecutor:
    """带重试和降级的工具执行器"""
    
    def __init__(self, max_retries=3, base_delay=1.0, timeout=10):
        self.max_retries = max_retries
        self.base_delay = base_delay
        self.timeout = timeout
        self.call_log = []
    
    def execute(self, tool_name, args, tool_fn):
        """执行工具调用，带重试机制"""
        last_error = None
        
        for attempt in range(1, self.max_retries + 1):
            try:
                start = time.time()
                result = tool_fn(**args)
                elapsed = time.time() - start
                
                # 记录成功调用
                self.call_log.append({
                    'tool': tool_name,
                    'attempt': attempt,
                    'status': 'success',
                    'time': f'{elapsed:.2f}s'
                })
                
                return {'status': 'success', 'data': result, 'attempts': attempt}
                
            except TimeoutError as e:
                last_error = e
                self.call_log.append({
                    'tool': tool_name,
                    'attempt': attempt,
                    'status': 'timeout',
                    'error': str(e)
                })
                
            except ValueError as e:
                last_error = e
                self.call_log.append({
                    'tool': tool_name,
                    'attempt': attempt,
                    'status': 'param_error',
                    'error': str(e)
                })
                # 参数错误通常重试没用，直接跳出
                break
                
            except Exception as e:
                last_error = e
                self.call_log.append({
                    'tool': tool_name,
                    'attempt': attempt,
                    'status': 'error',
                    'error': str(e)
                })
            
            # 指数退避
            delay = self.base_delay * (2 ** (attempt - 1))
            print(f"    ⚠️ 第{attempt}次失败，{delay}秒后重试...")
            time.sleep(0.1)  # 演示用，实际是 delay 秒
        
        # 所有重试都失败 → 优雅降级
        fallback = self._fallback(tool_name, args)
        return {'status': 'fallback', 'data': fallback, 'error': str(last_error)}
    
    def _fallback(self, tool_name, args):
        """优雅降级策略"""
        return f"[降级响应] 工具 {tool_name} 暂时不可用，已使用缓存数据"


# 模拟测试
print("=" * 50)
print("🧪 错误处理与重试机制演示")
print("=" * 50)

# 模拟工具1：间歇性超时
call_count = 0
def flaky_weather(city):
    global call_count
    call_count += 1
    if call_count % 3 != 0:  # 前2次失败
        raise TimeoutError(f"天气API超时 (城市: {city})")
    return {"city": city, "temp": "25°C", "condition": "晴"}

# 模拟工具2：参数错误
def strict_calculate(expression):
    if not expression or len(expression) < 3:
        raise ValueError("表达式不能为空或太短")
    return f"{expression} = 42"

executor = RobustToolExecutor(max_retries=3)

print("\n测试1: 间歇性超时（第3次成功）")
result = executor.execute('get_weather', {'city': '北京'}, flaky_weather)
print(f"结果: {result}")

print("\n测试2: 参数错误（不重试，直接降级）")
result = executor.execute('calculate', {'expression': ''}, strict_calculate)
print(f"结果: {result}")

print("\n测试3: 调用日志")
for log in executor.call_log:
    print(f"  {log}")

## 🔍 4. 可观测性设计

生产级Agent必须能"看见"自己在做什么。可观测性三支柱：

| 支柱 | 内容 | 工具 |
|------|------|------|
| **日志（Logging）** | 每步操作记录 | Python logging / Loguru |
| **追踪（Tracing）** | 请求链路追踪 | LangSmith / LangFuse |
| **指标（Metrics）** | 性能数据统计 | Prometheus / 自定义 |

In [ ]:
# 可观测性演示：Agent调用追踪

class AgentTracer:
    """简单的Agent调用追踪器"""
    
    def __init__(self):
        self.traces = []
        self.current_trace = None
    
    def start_trace(self, trace_id, user_message):
        self.current_trace = {
            'trace_id': trace_id,
            'user_message': user_message,
            'start_time': datetime.now().isoformat(),
            'spans': [],
            'status': 'running'
        }
    
    def add_span(self, name, span_type, data=None):
        if self.current_trace:
            self.current_trace['spans'].append({
                'name': name,
                'type': span_type,
                'timestamp': datetime.now().isoformat(),
                'data': data
            })
    
    def end_trace(self, status='success'):
        if self.current_trace:
            self.current_trace['status'] = status
            self.current_trace['end_time'] = datetime.now().isoformat()
            self.traces.append(self.current_trace)
            self.current_trace = None
    
    def report(self):
        for trace in self.traces:
            print(f"\n{'='*50}")
            print(f"📋 Trace: {trace['trace_id']}")
            print(f"   用户: {trace['user_message'][:50]}...")
            print(f"   状态: {trace['status']}")
            print(f"   耗时: {trace['start_time']} → {trace.get('end_time','')}")
            print(f"   步骤:")
            for span in trace['spans']:
                icon = '🧠' if span['type']=='llm' else '🔧' if span['type']=='tool' else '💬'
                print(f"     {icon} {span['name']}")


tracer = AgentTracer()

# 模拟一个完整的Agent调用链
tracer.start_trace('trace-001', '北京天气怎么样？如果温度超过30度就通知我')
tracer.add_span('意图理解', 'llm', data={'需要工具': ['get_weather', 'send_notification']})
tracer.add_span('调用天气API', 'tool', data={'tool': 'get_weather', 'city': '北京'})
tracer.add_span('天气结果分析', 'llm', data={'温度': '35°C', '结论': '超过30度'})
tracer.add_span('发送通知', 'tool', data={'tool': 'send_notification', 'channel': 'wechat'})
tracer.add_span('生成回复', 'llm', data={'回复': '北京35°C，已发通知'})
tracer.end_trace('success')

tracer.start_trace('trace-002', '你好')
tracer.add_span('意图理解', 'llm', data={'需要工具': []})
tracer.add_span('直接回复', 'llm', data={'回复': '你好！有什么可以帮你的？'})
tracer.end_trace('success')

tracer.report()

print("\n\n💡 生产环境建议使用 LangSmith 或 LangFuse 做完整的调用追踪")

## 🧩 5. 主流Agent框架对比

选择框架时要考虑：团队技术栈、复杂度需求、延迟要求、社区生态。

In [ ]:
# 主流Agent框架对比
fig, ax = plt.subplots(1, 1, figsize=(14, 8))
ax.axis('off')

frameworks = [
    {
        'name': 'LangChain + LangGraph',
        'pros': '生态最丰富、社区活跃、集成多',
        'cons': '抽象层厚、调试困难、版本更新快',
        'best_for': '快速原型、多工具集成',
        'color': '#42A5F5'
    },
    {
        'name': 'OpenAI Agents SDK',
        'pros': '官方支持、简洁直观、与GPT深度集成',
        'cons': '绑定OpenAI生态、灵活性有限',
        'best_for': 'GPT为主的简单Agent',
        'color': '#10A37F'
    },
    {
        'name': 'AutoGen (Microsoft)',
        'pros': '多Agent协作优秀、灵活',
        'cons': '学习曲线陡、调试复杂',
        'best_for': '多Agent对话/协作场景',
        'color': '#F25022'
    },
    {
        'name': 'CrewAI',
        'pros': '角色定义直观、团队协作概念清晰',
        'cons': '功能相对简单、社区较小',
        'best_for': '角色扮演式多Agent',
        'color': '#7B68EE'
    },
    {
        'name': '自研轻量框架',
        'pros': '完全可控、无依赖、按需定制',
        'cons': '开发成本高、需自己处理边界',
        'best_for': '特定业务、高定制需求',
        'color': '#FF9800'
    },
]

ax.set_title('主流 Agent 框架对比', fontsize=16, fontweight='bold', pad=20)

# 表头
headers = ['框架', '优点', '缺点', '最适合']
col_x = [0.02, 0.28, 0.58, 0.82]
col_w = [0.24, 0.28, 0.22, 0.16]

for j, header in enumerate(headers):
    ax.text(col_x[j] + col_w[j]/2, 0.95, header, ha='center', va='center',
            fontsize=13, fontweight='bold', color='#333')

ax.axhline(y=0.92, xmin=0.02, xmax=0.98, color='#333', linewidth=1.5)

for i, fw in enumerate(frameworks):
    y = 0.87 - i * 0.17
    
    # 框架名
    rect = patches.FancyBboxPatch((col_x[0], y-0.06), col_w[0], 0.12,
                                    boxstyle="round,pad=0.02",
                                    facecolor=fw['color'], alpha=0.15,
                                    edgecolor=fw['color'], linewidth=1.5)
    ax.add_patch(rect)
    ax.text(col_x[0] + col_w[0]/2, y, fw['name'], ha='center', va='center',
            fontsize=11, fontweight='bold', color=fw['color'])
    
    # 优点/缺点/最适合
    ax.text(col_x[1], y, f'✅ {fw["pros"]}', va='center', fontsize=9)
    ax.text(col_x[2], y, f'❌ {fw["cons"]}', va='center', fontsize=9)
    ax.text(col_x[3] + col_w[3]/2, y, fw['best_for'], ha='center', va='center',
            fontsize=10, color=fw['color'], fontweight='bold')

plt.tight_layout()
plt.show()

## ✏️ 课堂练习（5分钟）

❶ **框架选择题**：你要做一个多Agent协作的销售助手，每个Agent负责不同产品线，推荐用哪个框架？
   A. LangChain   B. AutoGen   C. CrewAI   D. 都可以

❷ **设计题**：设计一个"智能客服Agent"的状态管理方案。要求：能记住用户历史投诉，能跨会话访问。

❸ **思考题**：为什么参数错误不适合自动重试？

In [ ]:
# 练习答案参考
print("=" * 50)
print("💡 课堂练习参考答案")
print("=" * 50)

print("""
❶ 答案: D. 都可以
   - AutoGen 和 CrewAI 天然支持角色协作
   - LangChain + LangGraph 也能通过图编排实现
   - 选择取决于团队经验和具体需求

❷ 参考方案:
   - 短期记忆: 上下文窗口（最近3轮对话）
   - 长期记忆: PostgreSQL 存储投诉记录
   - 检索层: 向量库存用户画像/历史摘要
   - 每次对话开始时从DB加载历史摘要放入系统提示

❸ 参数错误不适合重试的原因:
   - 参数错误说明LLM理解有偏差
   - 同样的输入重试会得到同样的错误参数
   - 正确做法: 把错误信息反馈给LLM，让它修正参数后重试
""")

## 📝 课后测试（15分钟）

❶ Agent框架的核心三要素是什么？
   A. LLM + 提示词 + 温度参数
   B. LLM大脑 + 工具库 + 执行引擎
   C. 输入层 + 处理层 + 输出层
   D. 前端 + 后端 + 数据库

❷ 以下哪种状态管理策略最适合Agent的"长期记忆"？
   A. Python dict
   B. JSON文件
   C. 向量数据库
   D. 环境变量

❸ 指数退避重试中，第1次等1秒，第2次等几秒？第3次呢？

❹ 简答：为什么Agent的可观测性很重要？至少列出两个原因。

❺ 设计题：你要为糖水店设计一个"订单管理Agent"，列出它需要的3个工具，并说明每个工具的作用。

**回复答案我帮你批改 ✅**

## 🎓 今日总结

### 核心要点

1. **Agent框架三要素**：LLM大脑（推理决策）+ 工具库（能力扩展）+ 执行引擎（流程控制）
2. **状态管理**：短期记忆用上下文，长期记忆用外部存储（DB/向量库）
3. **错误处理**：指数退避重试 + 参数错误特殊处理 + 优雅降级 + 人工兜底
4. **可观测性**：日志 + 追踪 + 指标，生产环境建议用 LangSmith/LangFuse
5. **框架选择**：没有最好的框架，只有最适合的（看团队、看场景、看复杂度）

### 知识脉络

```
Day1 Agent架构 → Day2 Function Calling → Day3 ReAct协作 → Day4 安全可控
                                                               ↓
                                   Day5 框架设计 ← 综合前面所有知识
                                       ↓
                               Day6 动手搭建实战
```

In [ ]:
# 第6周学习进度
fig, ax = plt.subplots(1, 1, figsize=(14, 5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 3)
ax.axis('off')
ax.set_title('第6周学习进度', fontsize=16, fontweight='bold', pad=15)

days = [
    (1, 'Day1', 'LLM Agent基本架构', '#4CAF50', True),
    (3, 'Day2', 'Function Calling详解', '#4CAF50', True),
    (5, 'Day3', 'ReAct模式与多Agent协作', '#4CAF50', True),
    (7, 'Day4', 'Agent安全与可控性', '#4CAF50', True),
    (9, 'Day5', 'Agent开发实战框架设计', '#4CAF50', True),
    (11, 'Day6', '搭建Function Calling Agent', '#4CAF50', True),
]

for x, label, topic, color, done in days:
    circle = plt.Circle((x, 1.5), 0.6, color=color, alpha=0.3 if not done else 0.8)
    ax.add_patch(circle)
    ax.text(x, 1.7, label, ha='center', va='center', fontsize=12, fontweight='bold')
    ax.text(x, 1.2, topic, ha='center', va='center', fontsize=9,
            color='#333' if done else '#999')
    if done:
        ax.text(x, 2.3, '✅', ha='center', fontsize=16)

for i in range(len(days) - 1):
    x1 = days[i][0] + 0.6
    x2 = days[i+1][0] - 0.6
    ax.plot([x1, x2], [1.5, 1.5], '-', color='#4CAF50', linewidth=2)

plt.tight_layout()
plt.show()

print("\n📊 进度：第6周/12 | Day5/7 | 大模型 - Agent与工具使用")

## 🔑 今日英文术语

| 术语 | 音标 | 中文释义 |
|------|------|----------|
| **Agent Framework** | /ˈeɪdʒənt ˈfreɪmwɜːk/ | Agent框架 |
| **State Management** | /steɪt ˈmænɪdʒmənt/ | 状态管理 |
| **Exponential Backoff** | /ˌekspəˈnenʃəl ˈbækɒf/ | 指数退避 |
| **Graceful Degradation** | /ˈɡreɪsfʊl ˌdeɡrəˈdeɪʃən/ | 优雅降级 |
| **Observability** | /əbˌzɜːvəˈbɪlɪti/ | 可观测性 |
| **Tracing** | /ˈtreɪsɪŋ/ | 链路追踪 |
| **Span** | /spæn/ | 追踪跨度（一次操作） |
| **Fallback** | /ˈfɔːlbæk/ | 降级/备选方案 |
| **Context Window** | /ˈkɒntekst ˈwɪndəʊ/ | 上下文窗口 |
| **LangSmith** | — | LangChain官方追踪平台 |

## 🔄 往期回顾

**问**：昨天我们学了Agent的安全可控性，护栏（Guardrails）和人工兜底有什么区别？

**答**：护栏是**事前预防**——在Agent执行前就设定好规则边界（如禁止调用某些工具、限制输出长度），是自动化的。人工兜底是**事后补救**——当Agent遇到不确定或高风险操作时，暂停等待人工确认。两者配合：护栏挡住大多数风险，人工兜底处理边界情况。

💡 明天 Day6 就是我们期待已久的实战日——从零搭建一个完整的 Function Calling Agent！今天学的框架设计思路会直接用到。